<div dir="rtl" align="right">
# تدريبُ EEGNet على بيانات التخيل الحركي

## نظرةٌ عامّةٌ
يُدرّبُ هذا الدفترُ بنيةَ **EEGNet** العصبيّةَ على بياناتِ التخيلِ الحركيِّ من مجموعةِ **BNCI2014-001** (الشخصُ الأول).

**مجموعةُ البيانات:** BNCI2014-001، 22 قناة، 250 Hz، 288 محاولة (فئتا left_hand و right_hand)

## ماذا يفعلُ هذا الدفترُ؟
- يَحمّلُ بياناتِ التخيلِ الحركيِّ ويُرشّحُها إلى نطاقِ 8-32 Hz
- يَبنى بنيةَ EEGNet في PyTorch
- يُدرّبُ النموذجَ لـ 50 حقبةً مع مُحسّنِ Adam
- يَرسمُ مُنحنى الخسارةِ ومصفوفةَ الالتباسِ تفاعليّاً

## المُخرجاتُ المُتوقّعةُ
- مُنحنى خسارةٍ يَنخفضُ عبرَ الحِقَب
- مصفوفةُ التباسٍ 2×2 تُظهرُ أداءَ التصنيف

## المُعاملاتُ الأساسيةُ
| المعامل | القيمة | المعنى |
|---------|--------|---------|
| fmin/fmax | 8/32 Hz | نطاقُ الترشيحِ |
| epochs | 50 | عددُ الحِقَب |
| lr | 0.001 | معدّلُ التعلّم |
| batch_size | 32 | حجمُ الدفعة |
</div>

<div dir="rtl" align="right">
## 1. تثبيتُ المكتباتِ
</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn torch

<div dir="rtl" align="right">
## 2. تحميلُ مجموعةِ بياناتِ MOABB
</div>

In [ ]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

y = np.array([0 if lab == 'left_hand' else 1 for lab in labels])
print(f"Data shape: {X.shape}")
print(f"Labels: {len(labels)}")
print(f"Classes: {np.unique(labels)}")

<div dir="rtl" align="right">
## 3. بناءُ نموذجِ EEGNet

EEGNet شبكةٌ عصبيّةٌ مُدمجةٌ بِـ ~3000 معاملٍ فقط. تَتكوّنُ من التفافٍ زمنيّ، التفافٍ مكانيٍّ عميق، والتفافٍ قابلٍ للفصل، وطبقةِ تصنيف.
</div>

In [ ]:
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_channels=22, n_samples=1001, n_classes=2, F1=8, D=2, F2=16, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, F1, (1, n_samples // 2), padding='same')
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1)
        self.batchnorm2 = nn.BatchNorm2d(F1 * D)
        self.activation = nn.ELU()
        self.pool1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(dropout)
        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding='same'),
            nn.Conv2d(F1 * D, F2, (1, 1)),
        )
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(dropout)
        dummy = torch.zeros(1, 1, n_channels, n_samples)
        out = self._features(dummy)
        self.classify = nn.Linear(out.view(-1).shape[0], n_classes)

    def _features(self, x):
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.depthwise(x)
        x = self.batchnorm2(x)
        x = self.activation(x)
        x = self.pool1(x)
        x = self.dropout1(x)
        x = self.separable(x)
        x = self.batchnorm3(x)
        x = self.activation(x)
        x = self.pool2(x)
        x = self.dropout2(x)
        return x

    def forward(self, x):
        x = self._features(x)
        x = x.view(x.size(0), -1)
        x = self.classify(x)
        return x

<div dir="rtl" align="right">
## 4. تدريبُ النموذجِ

نُقسّمُ البياناتِ 80/20، ثمّ نُدرّبُ النموذجَ لـ 50 حقبة.
</div>

In [ ]:
from sklearn.model_selection import train_test_split
import torch

torch.manual_seed(42)
np.random.seed(42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

n_channels = X.shape[1]
n_samples = X.shape[2]
model = EEGNet(n_channels=n_channels, n_samples=n_samples, n_classes=2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

losses = []
batch_size = 32
for epoch in range(50):
    perm = torch.randperm(X_train_t.shape[0])
    epoch_loss = 0.0
    n_batches = 0
    for start in range(0, X_train_t.shape[0], batch_size):
        idx = perm[start:start + batch_size]
        batch_x = X_train_t[idx]
        batch_y = y_train_t[idx]
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/50 - loss: {avg_loss:.4f}")

model.eval()
with torch.no_grad():
    outputs = model(X_test_t)
    _, predicted = torch.max(outputs, 1)
accuracy = np.mean(predicted.numpy() == y_test_t.numpy())
print(f"Final accuracy: {accuracy:.4f}")

<div dir="rtl" align="right">
## 5. رسمٌ تفاعليٌّ

### علامَ تُلاحظُ؟
- مُنحنى الخسارةِ يَنخفضُ بِشكلٍ مُتدرّج
- مصفوفةُ الالتباسِ تُظهرُ التنبؤاتِ الصحيحةَ والخاطئة
</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_t.numpy(), predicted.numpy(), labels=[0, 1])

fig = make_subplots(rows=1, cols=2, subplot_titles=('Training Loss', 'Confusion Matrix'))

fig.add_trace(go.Scatter(x=list(range(1, 51)), y=losses, mode='lines+markers', name='Loss'), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=cm, x=['left_hand', 'right_hand'], y=['left_hand', 'right_hand'],
    colorscale='Blues', text=cm.astype(str), texttemplate='%{text}',
    showscale=False
), row=1, col=2)

fig.update_layout(title=f'EEGNet Training (accuracy={accuracy:.4f})', width=1000, height=450)
fig.update_xaxes(title_text='Epoch', row=1, col=1)
fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_xaxes(title_text='Predicted', row=1, col=2)
fig.update_yaxes(title_text='True', row=1, col=2)
fig.show()

<div dir="rtl" align="right">
## خلاصةٌ
- EEGNet شبكةٌ مُدمجةٌ تَتعلّمُ السماتِ من الإشارةِ الخام
- ~3000 معاملٍ فقط، مناسبٌ لِبياناتِ EEG الصغيرة
- تَتدرّبُ في دقائقَ على CPU
- تَتفوّقُ عادةً على تعلّمِ الآلةِ الكلاسيكيِّ معَ السماتِ اليدويّة
</div>